# Hafta 5 — Çoklu Ajan Sistemleri ve Durum Yönetimi

> [`src/multi_agent/`](../src/multi_agent/) **için kaynak not defteri.**

## Öğrenme hedefleri

1. Çoklu ajan topolojilerini (hiyerarşik, eşler arası, kara tahta) karşılaştırmak ve bir görev için doğru olanı seçmek.
2. Ortak durumu yapılandırılmış bir kara tahta olarak modellemek ve içerik özeti (hash) yoluyla döngü tespiti uygulamak.
3. Bir ajanın kod yazdığı, bir diğerinin çalıştırdığı ve üçüncüsünün hataları teşhis ettiği, kendini düzelten bir Kodlayıcı–Yürütücü–Eleştirmen sistemi kurmak.
4. Döngünün sonlandığını kanıtlamak ve saf tur sayımının neden yetersiz olduğunu açıklamak.
5. Epizodik belleği anlamsal bellekten ayırt etmek ve her birinin bir ajan sisteminde nereye ait olduğunu açıklamak.

## 1. Neden birden fazla ajan?

Tek bir ReAct ajanı, problemin tamamını tek bir bağlamda ve tek bir kişilikte tutmak zorundadır. Çoklu ajan sistemleri bir problemi, her biri odaklı bir isteme ve dar bir sorumluluğa sahip *özelleşmiş* ajanlara dağıtır. Faydaları:

- **Sorumlulukların ayrılması.** Kod yazmak için optimize edilmiş bir "Kodlayıcı" istemi, hata bulmak için optimize edilmiş bir "Eleştirmen" isteminden farklıdır. Tek bir ajan her ikisinde birden en iyi olamaz.
- **Çekişmeli sağlamlık.** Kodu yazmamış bir Eleştirmen'in onu savunmaya yönelik batık maliyet önyargısı yoktur. Bu, kod incelemesinin çoklu ajan karşılığıdır.
- **Sınırlı bağlam.** Her ajan yalnızca ihtiyacı olanı görür; bu da tek tek bağlamları küçük ve odaklı tutar.

Bedeli **koordinasyon karmaşıklığıdır**: durum paylaşılmalı, turlar sıralanmalı ve sistemin *sonlanması garanti edilmelidir*.

## 2. Topolojiler

| Topoloji | Yapı | Ne zaman kullanılır |
|----------|-----------|-------------|
| **Hiyerarşik** | Bir yönetici ajan işi işçi ajanlara devreder ve sonuçları toplar | Açık bir ayrıştırması olan görevler (araştırma → bölümler → sentez) |
| **Eşler arası** | Ajanlar merkezî bir otorite olmadan doğrudan konuşur | Müzakere, tartışma, uzlaşı arayışı |
| **Kara tahta** | Ajanlar ortak bir durum nesnesini okur/yazar; bir denetleyici onları sıralar | Ortak bir öğe üzerinde yinelemeli iyileştirme (bizim durumumuz) |

Kendini düzelten kodlayıcımız **kara tahta** desenini kullanır: ortak bir `AgentState` nesnesi geçerli kodu, son hatayı ve sohbet geçmişini tutar. Bir denetleyici (`SelfCorrectingCoder`) Kodlayıcı'yı, Yürütücü'yü ve Eleştirmen'i sıralar.

## 3. Ortak durum ve döngü tespiti

Çoklu ajan sistemlerindeki en zor tek problem **sonlanmayan döngüdür**: iki ajanın iki konum arasında salınması ya da aynı bozuk öğeyi tekrar tekrar üretmesi.

Saf tur sayımı (`max_iterations`) en kötü durumu sınırlar ama bütçeyi döngülerde harcar. Daha iyi bir koruma: **her turda durumun verimli içeriğinin özetini (hash) al; bir özet tekrarlanırsa dur.**

Durum [`src/multi_agent/state.py`](../src/multi_agent/state.py) dosyasında yaşar.

In [ ]:
from src.multi_agent.state import AgentState

state = AgentState(task="merge sort uygula")

# Tur 1: Kodlayıcı başarısız olan bir kod yazar.
state.code = "def merge_sort(x): return sorted(x)  # yer tutucu"
state.last_error = "AssertionError: kararlılık testi başarısız"
print("bu durum ilk kez görüldüğünde döngü mü?", state.detect_loop())   # False — yeni

# Tur 2: Kodlayıcı *tamamen aynı* kodu ve *tamamen aynı* hatayı üretir.
print("tekrarda döngü mü?                     ", state.detect_loop())   # True — tıkandı


In [ ]:
# Ancak gerçek ilerleme dedektörü sıfırlar.
state2 = AgentState(task="t")
state2.code = "v1"; state2.last_error = "hata A"
print("v1/hataA ->", state2.detect_loop())   # False

state2.code = "v2"; state2.last_error = "hata B"   # Kodlayıcı revize etti → ilerleme
print("v2/hataB ->", state2.detect_loop())   # False


Parmak izi `(code, last_error)` ikilisinin özetini alır, çünkü bunlar durumun *verimli* bileşenleridir. Sohbet geçmişi monoton olarak büyür ve asla tekrarlanmaz; bu yüzden onun özetini almak dedektörü işe yaramaz kılar. Neyin özetinin alınacağını seçmek bir tasarım kararıdır: durgunluğu tıkanmış bir döngüyü tanımlayan öğenin özetini alın.

## 4. Kodlayıcı–Yürütücü–Eleştirmen topolojisi

```
   görev
    │
    ▼
 ┌───────────┐  kod  ┌──────────┐ geçti/kaldı+stderr ┌──────────┐
 │ Kodlayıcı │──────▶│ Yürütücü │───────────────────▶│ (denetim)│
 └───────────┘       └──────────┘                    └────┬─────┘
    ▲                                                      │ kaldı
    │ revise(kod, hata, eleştiri)                          ▼
    │                                                  ┌────────────┐
    └──────────────── eleştiri ────────────────────────│ Eleştirmen │
                                                        └────────────┘
```

- **Kodlayıcı** (LLM): Python yazar ve revize eder.
- **Yürütücü** (LLM *değil*): kodu zaman aşımı olan bir alt süreçte çalıştırır, stdout/stderr/returncode değerlerini yakalar.
- **Eleştirmen** (LLM): başarısız kodu ve hata izini alarak kök nedeni teşhis eder ve doğal dilde bir düzeltme stratejisi önerir.

Yürütücü'nün deterministik olması (LLM olmaması) bilinçlidir: kod yürütme, bir modelin kodun çalışıp çalışmayacağına dair *görüşü* değil, gerçeğin ta kendisi olmalıdır.

In [ ]:
# Yalıtılmış hâlde Yürütücü — saf alt süreç yürütmesi, LLM yok.
from src.multi_agent.agents import ExecutorAgent

executor = ExecutorAgent(timeout_seconds=5.0)

good = '''
def merge_sort(a):
    if len(a) <= 1: return a
    mid = len(a)//2
    L, R = merge_sort(a[:mid]), merge_sort(a[mid:])
    out, i, j = [], 0, 0
    while i < len(L) and j < len(R):
        if L[i] <= R[j]: out.append(L[i]); i += 1
        else: out.append(R[j]); j += 1
    return out + L[i:] + R[j:]

import random
data = list(range(1000)); random.shuffle(data)
assert merge_sort(data) == sorted(data)
print("OK: merge sort 1000 eleman üzerinde doğrulandı")
'''

result = executor.run(good)
print("geçti:     ", result.passed)
print("returncode:", result.returncode)
print("stdout:    ", result.stdout.strip())


In [ ]:
# Başarısız bir program → Yürütücü hata izini yakalar.
bad = '''
def merge_sort(a):
    return a[::-1]   # yanlış: sadece ters çevirir
import random
data = list(range(100)); random.shuffle(data)
assert merge_sort(data) == sorted(data), "sıralı değil!"
print("OK")
'''
result = executor.run(bad)
print("geçti:", result.passed)
print("stderr sonu:", result.stderr.strip().splitlines()[-1])


## 5. Kendini düzelten döngünün tamamı

`SelfCorrectingCoder` ([`src/multi_agent/orchestrator.py`](../src/multi_agent/orchestrator.py)), üç ajanı döngü dedektörü ve yineleme bütçesiyle birbirine bağlar. Not defteri API anahtarı olmadan çalışsın diye bunu senaryolu LLM'lerle gösteriyoruz; gerçek sürüm `build_client(...)` kullanır.

In [ ]:
from src.multi_agent.orchestrator import SelfCorrectingCoder

class FakeLLM:
    def __init__(self, scripted): self.scripted = list(scripted)
    def complete(self, messages, **kw): return self.scripted.pop(0)

# Kodlayıcı önce hatalı bir sürüm, ardından (eleştiriden sonra) doğru bir sürüm yazar.
buggy = '''```python
def is_palindrome(s):
    return s == s   # hata: her zaman True
assert is_palindrome("racecar")
assert not is_palindrome("hello"), "palindrom olmayanı reddetmeli"
print("OK")
```'''

fixed = '''```python
def is_palindrome(s):
    return s == s[::-1]
assert is_palindrome("racecar")
assert not is_palindrome("hello")
print("OK: palindrom denetleyicisi doğrulandı")
```'''

coder_llm  = FakeLLM([buggy, fixed])              # önce initial(), sonra revise()
critic_llm = FakeLLM(["Fonksiyon s'yi tersiyle değil kendisiyle karşılaştırıyor. s'yi s[::-1] ile karşılaştırın."])

# Rol başına ayrı LLM verilebilir; gerçek kullanımda tek bir build_client(...) yeterlidir.
coder = SelfCorrectingCoder(max_iterations=5, timeout_seconds=5.0,
                            coder_llm=coder_llm, critic_llm=critic_llm)

result = coder.solve("Öz-denetimli bir is_palindrome(s) fonksiyonu yaz.")
print("başarı:       ", result.success)
print("yinelemeler:  ", result.iterations)
print("halted_reason:", result.halted_reason)
print("--- nihai kod ---")
print(result.final_code)


**Ne oldu:** 1. yineleme hatalı kodu çalıştırdı, Yürütücü assertion hatasını bildirdi, Eleştirmen `s == s` hatasını teşhis etti, Kodlayıcı kodu `s == s[::-1]` olarak revize etti ve 2. yineleme geçti. Döngü 2 yinelemede `halted_reason="success"` ile sonlandı.

## 6. Sonlanma garantileri

$t$ yinelemesindeki durum $s_t = (c_t, e_t)$ olsun — kod ve en son hata. Orkestratör, görülmüş durum özetlerinden oluşan bir $\mathcal{H}$ kümesi tutar. İki garanti:

1. **Katı sınır.** Döngü, davranıştan bağımsız olarak en fazla `max_iterations` kez çalışır.
2. **Döngü sınırı.** $\text{hash}(s_t) \in \mathcal{H}$ ise döngü `halted_reason="loop_detected"` ile hemen durur.

Birlikte bunlar sistemin *her zaman durmasını* ve ilerleme kaydetmediğini tespit ettiğinde — tüm bütçeyi bir döngüde yakmak yerine — *erken* durmasını sağlar.

In [ ]:
# Döngü tespitini göster: sürekli aynı bozuk kodu üreten bir Kodlayıcı.
stuck_code = '''```python
assert 1 == 2, "her seferinde aynı şekilde başarısız olur"
```'''
coder_llm  = FakeLLM([stuck_code, stuck_code, stuck_code, stuck_code])
critic_llm = FakeLLM(["1 == 2 assertion'ı imkânsızdır."] * 4)

coder = SelfCorrectingCoder(max_iterations=10, timeout_seconds=5.0,
                            coder_llm=coder_llm, critic_llm=critic_llm)

result = coder.solve("imkânsız görev")
print("halted_reason:", result.halted_reason, "—", result.iterations, "yinelemeden sonra")


Döngünün 10 değil **2** yinelemeden sonra durduğuna dikkat edin — dedektör tekrarlanan `(code, error)` parmak izini yakaladı ve işlemi iptal ederek 8 boşa LLM çağrısından tasarruf etti.

## 7. Bellek: epizodik ve anlamsal

Bir ajan sisteminde iki tür bellek bulunur:

- **Epizodik bellek** — olup bitenlerin *izi*: hangi eylemlerin yapıldığı, hangi gözlemlerin döndüğü. `AgentState.history` epizodiktir. Kısa ömürlüdür (görev başına) ve döngü tespitini ve hata ayıklamayı destekler.
- **Anlamsal bellek** — tek bir görevden daha uzun yaşayan *damıtılmış bilgi*: "bu derlem 'transformer' sözcüğünü elektrik cihazı değil mimari anlamında kullanır." Anlamsal bellek genellikle ajanın içinden erişim yaptığı bir vektör deposudur (Hafta 2!).

Olgun bir ajan, bir görevin sonunda öne çıkan epizodik olayları anlamsal belleğe yazar — böylece bu dersin ajan ve erişim yarıları arasındaki döngüyü kapatır.

## 8. Alıştırmalar

1. **Bir Planlayıcı ekleyin.** Görevi, Kodlayıcı başlamadan *önce* alt görevlere ayrıştıran dördüncü bir ajan ekleyin. Çok parçalı görevlerde başarı oranını artırıyor mu?
2. **Korumalı alanın sağlamlaştırılması.** Yürütücü kodu bir konteynerde değil, bir alt süreçte çalıştırır. Onu Docker tabanlı bir yürütücüyle değiştirin ve tehdit modelini (dosya sistemi, ağ, kaynak tüketimi) tartışın.
3. **Eleştirmen kalibrasyonu.** Eleştirmen bazen yanlış teşhis koyar. Eleştirmen'in çıktısına bir güven skoru ekleyin ve Kodlayıcı'nın revizyonları buna göre ağırlıklandırmasını sağlayın. Bir hatalı program paketi üzerinde ölçün.
4. **Kalıcı anlamsal bellek.** Çözülen her görevden sonra tek satırlık bir dersi ("merge sort bir kararlılık testine ihtiyaç duyar") bir Chroma deposuna gömün. Bir sonraki görevin başında ilgili dersleri getirin.

## 9. Çıkarımlar

- Çoklu ajan sistemleri, koordinasyon karmaşıklığını sorumlulukların ayrılması ve çekişmeli sağlamlık karşılığında takas eder. Tek bir kişilik/bağlam işi iyi yapamıyorsa bunları kullanın.
- **Kara tahta** deseni — ortak durum, sıralanmış ajanlar — yinelemeli iyileştirme için temel topolojidir.
- **Döngü tespiti için verimli durumun özetini alın.** Tur sayımı en kötü durumu sınırlar; içerik özeti tıkanmaları erkenden yakalar.
- Yürütücü bir LLM değil, **gerçeğin kendisi** olmalıdır. Kod ya çalışır ya çalışmaz.
- Epizodik (görev başına iz) ve anlamsal (kalıcı bilgi) belleği ayırt edin; ikincisi bir vektör deposudur ve Hafta 2–3'e geri bağlanır.

➡ Gelecek hafta her şeyi — MCP, RAG, yeniden sıralama, ReAct ve çoklu ajan durumu — tek bir otonom Akademik Araştırma Asistanında birleştiriyoruz.